# [title] One-state Model: a Simulated Experiment

**Question.** Can `motor_sbi.experiments` simulate an Experiment from the one-state Model,
save it with its Ground Truth kept apart, and load it back unchanged, and how long does
that take for the sizes BayesFlow will need?

**Scope.** Step C3a of [issue #12](https://github.com/opherdonchin/MotorAdpatationSBI/issues/12).
The function is `simulate_experiment` in
[`src/motor_sbi/experiments.py`](../../src/motor_sbi/experiments.py); where it fits:
[decision 0006](../../docs/decisions/0006-models-tasks-and-designs.md). Terms:
[glossary](../../docs/glossary.md).

**Checks:**

1. *Round trip:* the Sittings and the Ground Truth loaded from disk equal those simulated.
2. *Blindness:* the saved Sittings hold only Conditions and Observations; parameter values
   and hidden states are only in the Ground Truth file.
3. *Prior through the pipeline:* the parameter values in the Ground Truth have the Prior's
   quantiles, so `simulate_experiment` hands the Prior Sampler's draws on unchanged.

In [ ]:
# [imports]
import json
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import special, stats

from designs import visuomotor_adaptation_experiment as des
from models import one_state as mdl
from motor_sbi import experiments

In [ ]:
# [config] Root seed, provenance, Prior and Schedule Design, and the size of the Experiment
root_seed = 106935205776254155769853502130119773734  # secrets.randbits(128)

# Provenance (decision 0006, AGENTS.md *Where things live*): the repository's top folder,
# the commit that produced the outputs (with -dirty if there were uncommitted changes),
# and where this notebook's output goes.
REPO = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
GIT_COMMIT = subprocess.check_output(["git", "describe", "--always", "--dirty"], text=True).strip()
OUT_DIR = REPO / "data" / "simulated" / "one_state_random_blocks_demo"

# The Prior and Schedule Design of decision 0005, from their stated 95% ranges.
z95 = stats.norm.ppf(0.975)
range_a, range_b = (0.75, 0.999), (0.01, 0.5)
range_sigma_epsilon, range_ratio, range_block_len = (1 / 15, 1 / 3), (1 / 15, 1 / 5), (10, 250)
prior = {
    "mu_logit_A": special.logit(range_a).mean(), "sigma_logit_A": np.ptp(special.logit(range_a)) / (2 * z95),
    "mu_logit_B": special.logit(range_b).mean(), "sigma_logit_B": np.ptp(special.logit(range_b)) / (2 * z95),
    "mu_log_sigma_epsilon": np.log(range_sigma_epsilon).mean(),
    "sigma_log_sigma_epsilon": np.ptp(np.log(range_sigma_epsilon)) / (2 * z95),
    "mu_log_ratio": np.log(range_ratio).mean(), "sigma_log_ratio": np.ptp(np.log(range_ratio)) / (2 * z95),
}
schedule_design = {
    "n_blocks_min": 5, "n_blocks_max": 16,
    "mu_log_block_len": np.log(range_block_len).mean(),
    "sigma_log_block_len": np.ptp(np.log(range_block_len)) / (2 * z95),
    "trial_type_concentration": 3.0,
}

n_sittings = 2_000  # large enough to check the Prior's quantiles, small enough to save quickly

In [ ]:
# [rng] One root generator; simulate_experiment spawns its own streams from it
rng = np.random.default_rng(root_seed)

## [simulate] Simulate and save

`simulate_experiment` draws each Sitting's parameter values from the Prior, its Schedule
from the Schedule Design, and its movements from the Simulator, using a separate stream
for each. `save_experiment` writes three files: `sittings.npz` (one row per Trial: Sitting,
Trial, `p`, `v`, `y`), `ground_truth.npz` (parameter values per Sitting, hidden plans per
Trial) and `record.json` (what produced the Experiment, including the seed and commit).

In [ ]:
# [simulate-and-save] Simulate the Experiment and save it, timing both
start = time.perf_counter()
sittings, ground_truth = experiments.simulate_experiment(
    rng, mdl, prior, des.sample_schedule, schedule_design, n_sittings
)
seconds_to_simulate = time.perf_counter() - start

record = {
    "model": mdl.__name__,
    "schedule_generator": f"{des.__name__}.sample_schedule",
    "prior": prior,
    "schedule_design": schedule_design,
    "n_sittings": n_sittings,
    "root_seed": str(root_seed),
    "git_commit": GIT_COMMIT,
}
start = time.perf_counter()
experiments.save_experiment(OUT_DIR, sittings, ground_truth, record)
seconds_to_save = time.perf_counter() - start

n_trials = sum(len(s["y"]) for s in sittings)
pd.Series({
    "Sittings": n_sittings,
    "Trials in all": n_trials,
    "seconds to simulate": round(seconds_to_simulate, 1),
    "milliseconds per Sitting": round(1000 * seconds_to_simulate / n_sittings, 2),
    "seconds to save": round(seconds_to_save, 1),
    "megabytes on disk": round(sum(f.stat().st_size for f in OUT_DIR.iterdir()) / 1e6, 1),
    "files": ", ".join(sorted(f.name for f in OUT_DIR.iterdir())),
})

## [check-1] Checks 1 and 2: round trip and blindness

In [ ]:
# [check-round-trip] Load everything back and compare
loaded = experiments.load_sittings(OUT_DIR)
truth, loaded_record = experiments.load_ground_truth(OUT_DIR)

same_sittings = len(loaded) == len(sittings) and all(
    set(a) == set(b) and all(np.array_equal(a[k], b[k]) for k in a) for a, b in zip(sittings, loaded)
)
same_truth = all(
    np.array_equal(truth[name], [t[name] for t in ground_truth]) for name in mdl.PARAMETERS
) and np.array_equal(truth["x"], np.concatenate([t["x"] for t in ground_truth]))
saved_columns = sorted(np.load(OUT_DIR / "sittings.npz").files)

assert same_sittings and same_truth and loaded_record == json.loads(json.dumps(record))
pd.Series({
    "Sittings equal after loading": same_sittings,
    "Ground Truth equal after loading": same_truth,
    "record equal after loading": loaded_record == json.loads(json.dumps(record)),
    "columns of sittings.npz": ", ".join(saved_columns),
    "columns of ground_truth.npz": ", ".join(sorted(truth)),
})

## [check-3] Check 3: the Prior survives the pipeline

The parameter values recorded in the Ground Truth should have the Prior's stated
central 95% intervals (decision 0005), as in `one_state_priors.ipynb`, here with only
2,000 draws, so to within about 0.01 to 0.02.

In [ ]:
# [check-prior] Quantiles of the Ground Truth's parameter values against the stated ranges
ratio = truth["sigma_eta"] / truth["sigma_epsilon"]
drawn = {"A": truth["A"], "B": truth["B"], "sigma_epsilon": truth["sigma_epsilon"], "ratio": ratio}
stated = {"A": range_a, "B": range_b, "sigma_epsilon": range_sigma_epsilon, "ratio": range_ratio}
pd.DataFrame({
    name: {
        "stated 2.5%": stated[name][0], "drawn 2.5%": np.quantile(x, 0.025),
        "stated 97.5%": stated[name][1], "drawn 97.5%": np.quantile(x, 0.975),
    }
    for name, x in drawn.items()
}).T.round(3)